In [1]:
import pandas as pd
import requests
import time
import re
from pathlib import Path
from bs4 import BeautifulSoup

/Library/Python/3.9/site-packages/urllib3/__init__.py:34: NotOpenSSLWarning: urllib3 v2.0 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

COMPANIES_PATH = DATA_DIR / "companies.csv"

PART2_METADATA_PATH = DATA_DIR / "part2_proxy_metadata.csv"
PART2_OUTPUT_PATH = DATA_DIR / "part2_proxy_disclosure_analysis.csv"
PART2_PROGRESS_PATH = DATA_DIR / "part2_proxy_disclosure_analysis_progress.csv"

START_YEAR = 2016
END_YEAR = 2024

In [7]:
SEC_HEADERS = {
    "User-Agent": "Lufei Chen lufeic22@seas.upenn.edu"
}

In [8]:
def get_sec_ticker_mapping():
    """
    Download SEC ticker-to-CIK mapping.
    """
    url = "https://www.sec.gov/files/company_tickers.json"
    
    response = requests.get(url, headers=SEC_HEADERS, timeout=30)
    response.raise_for_status()
    
    data = response.json()
    
    rows = []
    for _, item in data.items():
        rows.append({
            "ticker_for_sec": item["ticker"],
            "company_name_sec": item["title"],
            "cik": str(item["cik_str"]).zfill(10)
        })
    
    return pd.DataFrame(rows)


ticker_map = get_sec_ticker_mapping()

ticker_map.head()

,ticker_for_sec,company_name_sec,cik
0,NVDA,NVIDIA CORP,0001045810
1,AAPL,Apple Inc.,0000320193
2,GOOGL,Alphabet Inc.,0001652044
3,MSFT,MICROSOFT CORP,0000789019
4,AMZN,AMAZON COM INC,0001018724


In [12]:
companies["ticker_for_sec"] = companies["ticker"].replace({
    "BRK.B": "BRK-B"
})

companies_sec = companies.merge(
    ticker_map[["ticker_for_sec", "cik", "company_name_sec"]],
    on="ticker_for_sec",
    how="left"
)

companies_sec[["ticker", "ticker_for_sec", "company_name", "cik", "company_name_sec"]].head(10)

,ticker,ticker_for_sec,company_name,cik,company_name_sec
0,MSFT,MSFT,Microsoft,0000789019,MICROSOFT CORP
1,AAPL,AAPL,Apple,0000320193,Apple Inc.
2,NVDA,NVDA,NVIDIA,0001045810,NVIDIA CORP
3,GOOGL,GOOGL,Alphabet,0001652044,Alphabet Inc.
4,META,META,Meta Platforms,0001326801,"Meta Platforms, Inc."
5,AVGO,AVGO,Broadcom,0001730168,Broadcom Inc.
6,CRM,CRM,Salesforce,0001108524,"Salesforce, Inc."
7,ORCL,ORCL,Oracle,0001341439,ORACLE CORP
8,IBM,IBM,IBM,0000051143,INTERNATIONAL BUSINESS MACHINES CORP
9,INTC,INTC,Intel,0000050863,INTEL CORP


In [13]:
def get_company_submissions(cik, max_retries=3):
    """
    Get SEC submissions metadata for one company using its CIK.
    """
    url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    
    for attempt in range(max_retries):
        try:
            response = requests.get(url, headers=SEC_HEADERS, timeout=30)
            
            if response.status_code == 200:
                return response.json()
            
            print(f"Attempt {attempt + 1}: failed {response.status_code} for CIK {cik}")
            time.sleep(5 * (attempt + 1))
        
        except Exception as e:
            print(f"Attempt {attempt + 1}: error for CIK {cik}: {e}")
            time.sleep(5 * (attempt + 1))
    
    return None

In [14]:
def extract_def14a_filings(submissions, start_year=2016, end_year=2024):
    """
    Extract DEF 14A proxy statement filings from SEC submissions metadata.
    """
    if submissions is None:
        return pd.DataFrame()
    
    recent = submissions.get("filings", {}).get("recent", {})
    
    if not recent:
        return pd.DataFrame()
    
    filings = pd.DataFrame({
        "accession_number": recent.get("accessionNumber", []),
        "filing_date": recent.get("filingDate", []),
        "report_date": recent.get("reportDate", []),
        "form": recent.get("form", []),
        "primary_document": recent.get("primaryDocument", [])
    })
    
    if filings.empty:
        return filings
    
    filings["filing_year"] = pd.to_datetime(filings["filing_date"], errors="coerce").dt.year
    
    proxy_filings = filings[
        (filings["form"] == "DEF 14A") &
        (filings["filing_year"].between(start_year, end_year))
    ].copy()
    
    return proxy_filings

In [15]:
def build_sec_document_url(cik, accession_number, primary_document):
    """
    Build the direct SEC filing document URL.
    """
    cik_no_zeros = str(int(cik))
    accession_no_dashes = accession_number.replace("-", "")
    
    return (
        f"https://www.sec.gov/Archives/edgar/data/"
        f"{cik_no_zeros}/{accession_no_dashes}/{primary_document}"
    )

In [16]:
msft = companies_sec[companies_sec["ticker"] == "MSFT"].iloc[0]

msft_cik = msft["cik"]
print(msft["ticker"], msft["company_name"], msft_cik)

msft_submissions = get_company_submissions(msft_cik)

msft_proxy = extract_def14a_filings(
    msft_submissions,
    start_year=START_YEAR,
    end_year=END_YEAR
)

msft_proxy

MSFT Microsoft 0000789019


,accession_number,filing_date,report_date,form,primary_document,filing_year
268,0001193125-24-242883,2024-10-24,2024-12-10,DEF 14A,d858775ddef14a.htm,2024
451,0001193125-23-259247,2023-10-19,2023-12-07,DEF 14A,d356108ddef14a.htm,2023
589,0001193125-22-270484,2022-10-27,2022-12-13,DEF 14A,d318171ddef14a.htm,2022
735,0001193125-21-298757,2021-10-14,2021-11-30,DEF 14A,d189481ddef14a.htm,2021
878,0001193125-20-272025,2020-10-19,2020-12-02,DEF 14A,d31295ddef14a.htm,2020


In [17]:
msft_proxy["filing_url"] = msft_proxy.apply(
    lambda row: build_sec_document_url(
        msft_cik,
        row["accession_number"],
        row["primary_document"]
    ),
    axis=1
)

msft_proxy[["filing_year", "filing_date", "form", "filing_url"]]

,filing_year,filing_date,form,filing_url
268,2024,2024-10-24,DEF 14A,https://www.sec.gov/Archives/edgar/data/789019...
451,2023,2023-10-19,DEF 14A,https://www.sec.gov/Archives/edgar/data/789019...
589,2022,2022-10-27,DEF 14A,https://www.sec.gov/Archives/edgar/data/789019...
735,2021,2021-10-14,DEF 14A,https://www.sec.gov/Archives/edgar/data/789019...
878,2020,2020-10-19,DEF 14A,https://www.sec.gov/Archives/edgar/data/789019...


In [18]:
metadata_rows = []

for idx, company in companies_sec.iterrows():
    ticker = company["ticker"]
    company_name = company["company_name"]
    sector = company["sector"]
    cik = company["cik"]
    
    print(f"\n[{idx + 1}/{len(companies_sec)}] {ticker} - {company_name}")
    
    if pd.isna(cik):
        print("  Missing CIK")
        metadata_rows.append({
            "ticker": ticker,
            "company_name": company_name,
            "sector": sector,
            "cik": None,
            "year": None,
            "document_type": "DEF 14A",
            "filing_date": None,
            "filing_url": None,
            "metadata_status": "missing_cik"
        })
        continue
    
    submissions = get_company_submissions(cik)
    proxy_filings = extract_def14a_filings(
        submissions,
        start_year=START_YEAR,
        end_year=END_YEAR
    )
    
    if proxy_filings.empty:
        print("  No DEF 14A found")
        metadata_rows.append({
            "ticker": ticker,
            "company_name": company_name,
            "sector": sector,
            "cik": cik,
            "year": None,
            "document_type": "DEF 14A",
            "filing_date": None,
            "filing_url": None,
            "metadata_status": "no_def14a_found"
        })
        continue
    
    for _, filing in proxy_filings.iterrows():
        filing_url = build_sec_document_url(
            cik,
            filing["accession_number"],
            filing["primary_document"]
        )
        
        metadata_rows.append({
            "ticker": ticker,
            "company_name": company_name,
            "sector": sector,
            "cik": cik,
            "year": int(filing["filing_year"]),
            "document_type": "DEF 14A",
            "filing_date": filing["filing_date"],
            "report_date": filing["report_date"],
            "accession_number": filing["accession_number"],
            "primary_document": filing["primary_document"],
            "filing_url": filing_url,
            "metadata_status": "found"
        })
    
    time.sleep(0.5)

part2_metadata = pd.DataFrame(metadata_rows)

part2_metadata.head()


[1/50] MSFT - Microsoft

[2/50] AAPL - Apple

[3/50] NVDA - NVIDIA

[4/50] GOOGL - Alphabet

[5/50] META - Meta Platforms
  No DEF 14A found

[6/50] AVGO - Broadcom

[7/50] CRM - Salesforce

[8/50] ORCL - Oracle

[9/50] IBM - IBM

[10/50] INTC - Intel

[11/50] BRK.B - Berkshire Hathaway

[12/50] JPM - JPMorgan Chase
  No DEF 14A found

[13/50] BAC - Bank of America
  No DEF 14A found

[14/50] WFC - Wells Fargo
  No DEF 14A found

[15/50] GS - Goldman Sachs
  No DEF 14A found

[16/50] MS - Morgan Stanley
  No DEF 14A found

[17/50] BLK - BlackRock
  No DEF 14A found

[18/50] SCHW - Charles Schwab

[19/50] AXP - American Express

[20/50] C - Citigroup
  No DEF 14A found

[21/50] LLY - Eli Lilly

[22/50] UNH - UnitedHealth Group

[23/50] JNJ - Johnson & Johnson

[24/50] ABBV - AbbVie

[25/50] MRK - Merck

[26/50] TMO - Thermo Fisher Scientific

[27/50] ABT - Abbott Laboratories

[28/50] PFE - Pfizer

[29/50] MDT - Medtronic

[30/50] BMY - Bristol-Myers Squibb

[31/50] AMZN - Amazon

[32/

,ticker,company_name,sector,cik,year,document_type,filing_date,report_date,accession_number,primary_document,filing_url,metadata_status
0,MSFT,Microsoft,Technology,0000789019,2024.0,DEF 14A,2024-10-24,2024-12-10,0001193125-24-242883,d858775ddef14a.htm,https://www.sec.gov/Archives/edgar/data/789019...,found
1,MSFT,Microsoft,Technology,0000789019,2023.0,DEF 14A,2023-10-19,2023-12-07,0001193125-23-259247,d356108ddef14a.htm,https://www.sec.gov/Archives/edgar/data/789019...,found
2,MSFT,Microsoft,Technology,0000789019,2022.0,DEF 14A,2022-10-27,2022-12-13,0001193125-22-270484,d318171ddef14a.htm,https://www.sec.gov/Archives/edgar/data/789019...,found
3,MSFT,Microsoft,Technology,0000789019,2021.0,DEF 14A,2021-10-14,2021-11-30,0001193125-21-298757,d189481ddef14a.htm,https://www.sec.gov/Archives/edgar/data/789019...,found
4,MSFT,Microsoft,Technology,0000789019,2020.0,DEF 14A,2020-10-19,2020-12-02,0001193125-20-272025,d31295ddef14a.htm,https://www.sec.gov/Archives/edgar/data/789019...,found


In [19]:
part2_metadata.to_csv(PART2_METADATA_PATH, index=False)

print(f"Saved metadata to: {PART2_METADATA_PATH}")
print(part2_metadata.shape)
part2_metadata["metadata_status"].value_counts()

Saved metadata to: ../data/part2_proxy_metadata.csv
(276, 12)


metadata_status
found              268
no_def14a_found      8
Name: count, dtype: int64

In [23]:
def download_sec_document(url, max_retries=3):
    """
    Download one SEC filing document.
    """
    if pd.isna(url) or url is None:
        return None
    
    for attempt in range(max_retries):
        try:
            response = requests.get(
                url,
                headers=SEC_HEADERS,
                timeout=60
            )
            
            if response.status_code == 200:
                return response.text
            
            print(f"Attempt {attempt + 1}: failed {response.status_code} for {url}")
            time.sleep(5 * (attempt + 1))
        
        except Exception as e:
            print(f"Attempt {attempt + 1}: error downloading {url}: {e}")
            time.sleep(5 * (attempt + 1))
    
    return None

In [25]:
def clean_sec_html(html):
    """
    Extract readable text from SEC filing HTML.
    """
    if html is None:
        return None
    
    soup = BeautifulSoup(html, "html.parser")
    
    # Remove non-text elements
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    
    text = soup.get_text(separator=" ")
    
    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [26]:
sample_filing = part2_metadata[
    (part2_metadata["ticker"] == "MSFT") &
    (part2_metadata["metadata_status"] == "found")
].iloc[0]

sample_url = sample_filing["filing_url"]

print(sample_filing["ticker"], sample_filing["year"])
print(sample_url)

sample_html = download_sec_document(sample_url)
sample_text = clean_sec_html(sample_html)

print("Text length:", len(sample_text) if sample_text else 0)
print(sample_text[:1000] if sample_text else "No text")

MSFT 2024.0
https://www.sec.gov/Archives/edgar/data/789019/000119312524242883/d858775ddef14a.htm
Text length: 347326
DEF 14A Table of Contents DEF 14A false 0000789019 0000789019 2023-07-01 2024-06-30 0000789019 2021-07-01 2022-06-30 0000789019 2020-07-01 2021-06-30 0000789019 2022-07-01 2023-06-30 0000789019 msft:FairValueOfEquityAwardsGrantedInTheYearAndUnvestedAsOfYearEndMember ecd:PeoMember 2023-07-01 2024-06-30 0000789019 msft:ChangeInFairValueOfOutstandingAndUnvestedEquityAwardsGrantedInPriorFiscalYearsMember ecd:PeoMember 2023-07-01 2024-06-30 0000789019 msft:ChangeInFairValueOfEquityAwardsVestedGrantedInAPreviousYearMember ecd:PeoMember 2023-07-01 2024-06-30 0000789019 msft:AmountsReportedAsStockAwardsInSctMember ecd:PeoMember 2023-07-01 2024-06-30 0000789019 msft:ChangeInFairValueOfEquityAwardsVestedGrantedInAPreviousYearMember ecd:NonPeoNeoMember 2023-07-01 2024-06-30 0000789019 msft:AmountsReportedAsStockAwardsInSctMember ecd:NonPeoNeoMember 2023-07-01 2024-06-30 0000789019 

In [27]:
THEME_KEYWORDS = {
    "diversity_inclusion": [
        "diversity", "inclusion", "inclusive", "equity", "belonging",
        "gender", "racial", "ethnic", "underrepresented"
    ],
    "sustainability_environment": [
        "sustainability", "sustainable", "climate", "environment",
        "emissions", "carbon", "renewable", "energy transition"
    ],
    "employees_human_capital": [
        "employees", "employee", "workforce", "talent", "human capital",
        "training", "retention", "culture", "compensation"
    ],
    "governance_ethics": [
        "governance", "ethics", "integrity", "compliance",
        "accountability", "risk oversight", "board oversight"
    ],
    "community_social_impact": [
        "community", "communities", "philanthropy", "charitable",
        "volunteer", "social impact", "stakeholder", "stakeholders"
    ],
    "risk_security": [
        "risk", "risks", "security", "cybersecurity", "privacy",
        "data security", "regulatory"
    ]
}

In [28]:
def count_theme_keywords(text, keyword_dict):
    """
    Count keyword mentions by theme.
    """
    counts = {theme: 0 for theme in keyword_dict}
    
    if text is None or pd.isna(text):
        return counts
    
    text_lower = text.lower()
    
    for theme, keywords in keyword_dict.items():
        count = 0
        
        for kw in keywords:
            count += text_lower.count(kw.lower())
        
        counts[theme] = count
    
    return counts

In [29]:
count_theme_keywords(sample_text, THEME_KEYWORDS)

{'diversity_inclusion': 137,
 'sustainability_environment': 147,
 'employees_human_capital': 676,
 'governance_ethics': 255,
 'community_social_impact': 58,
 'risk_security': 358}

In [30]:
FORWARD_LOOKING_WORDS = [
    "future", "strategy", "strategic", "long-term", "growth",
    "innovation", "opportunity", "commitment", "continue", "plan"
]

RISK_WORDS = [
    "risk", "uncertain", "uncertainty", "challenge", "compliance",
    "regulatory", "litigation", "security", "privacy"
]


def count_word_list(text, word_list):
    """
    Count simple word or phrase occurrences from a list.
    """
    if text is None or pd.isna(text):
        return 0
    
    text_lower = text.lower()
    return sum(text_lower.count(word.lower()) for word in word_list)

In [31]:
print("Forward-looking:", count_word_list(sample_text, FORWARD_LOOKING_WORDS))
print("Risk language:", count_word_list(sample_text, RISK_WORDS))

Forward-looking: 537
Risk language: 316


In [34]:
metadata_to_process = part2_metadata[
    part2_metadata["metadata_status"] == "found"
]

part2_rows = []

for idx, row in metadata_to_process.iterrows():
    ticker = row["ticker"]
    year = row["year"]
    filing_url = row["filing_url"]
    
    print(f"[{len(part2_rows) + 1}/{len(metadata_to_process)}] {ticker} {year}")
    
    html = download_sec_document(filing_url)
    clean_text = clean_sec_html(html)
    
    word_count = 0 if clean_text is None else len(clean_text.split())
    text_length = 0 if clean_text is None else len(clean_text)
    
    theme_counts = count_theme_keywords(clean_text, THEME_KEYWORDS)
    
    result = row.to_dict()
    result.update({
        "clean_text": clean_text,
        "word_count": word_count,
        "text_length": text_length,
        "download_status": "success" if clean_text and len(clean_text) > 500 else "failed_or_too_short",
        "forward_looking_count": count_word_list(clean_text, FORWARD_LOOKING_WORDS),
        "risk_language_count": count_word_list(clean_text, RISK_WORDS)
    })
    
    result.update(theme_counts)
    part2_rows.append(result)
    
    if len(part2_rows) % 10 == 0:
        progress_df = pd.DataFrame(part2_rows)
        progress_df.to_csv(PART2_PROGRESS_PATH, index=False)
        print(f"Progress saved to {PART2_PROGRESS_PATH}")
    
    time.sleep(1)

part2_df = pd.DataFrame(part2_rows)
part2_df.head()

[1/268] MSFT 2024.0
[2/268] MSFT 2023.0
[3/268] MSFT 2022.0
[4/268] MSFT 2021.0
[5/268] MSFT 2020.0
[6/268] AAPL 2024.0
[7/268] AAPL 2023.0
[8/268] AAPL 2022.0
[9/268] AAPL 2021.0
[10/268] AAPL 2020.0
Progress saved to ../data/part2_proxy_disclosure_analysis_progress.csv
[11/268] AAPL 2019.0
[12/268] AAPL 2017.0
[13/268] AAPL 2017.0
[14/268] AAPL 2016.0
[15/268] NVDA 2024.0
[16/268] NVDA 2023.0
[17/268] NVDA 2022.0
[18/268] NVDA 2022.0
[19/268] NVDA 2021.0
[20/268] NVDA 2020.0
Progress saved to ../data/part2_proxy_disclosure_analysis_progress.csv
[21/268] GOOGL 2024.0
[22/268] GOOGL 2023.0
[23/268] AVGO 2024.0
[24/268] AVGO 2023.0
[25/268] AVGO 2022.0
[26/268] AVGO 2021.0
[27/268] AVGO 2020.0
[28/268] AVGO 2019.0
[29/268] CRM 2024.0
[30/268] ORCL 2024.0
Progress saved to ../data/part2_proxy_disclosure_analysis_progress.csv
[31/268] ORCL 2023.0
[32/268] ORCL 2022.0
[33/268] ORCL 2021.0
[34/268] ORCL 2020.0
[35/268] ORCL 2019.0
[36/268] ORCL 2018.0
[37/268] ORCL 2017.0
[38/268] ORCL 2016

,ticker,company_name,sector,cik,year,document_type,filing_date,report_date,accession_number,primary_document,...,text_length,download_status,forward_looking_count,risk_language_count,diversity_inclusion,sustainability_environment,employees_human_capital,governance_ethics,community_social_impact,risk_security
0,MSFT,Microsoft,Technology,0000789019,2024.0,DEF 14A,2024-10-24,2024-12-10,0001193125-24-242883,d858775ddef14a.htm,...,347326,success,537,316,137,147,676,255,58,358
1,MSFT,Microsoft,Technology,0000789019,2023.0,DEF 14A,2023-10-19,2023-12-07,0001193125-23-259247,d356108ddef14a.htm,...,347076,success,516,268,174,129,741,243,50,294
2,MSFT,Microsoft,Technology,0000789019,2022.0,DEF 14A,2022-10-27,2022-12-13,0001193125-22-270484,d318171ddef14a.htm,...,314638,success,550,235,217,148,681,223,65,235
3,MSFT,Microsoft,Technology,0000789019,2021.0,DEF 14A,2021-10-14,2021-11-30,0001193125-21-298757,d189481ddef14a.htm,...,353635,success,664,208,225,78,824,233,61,178
4,MSFT,Microsoft,Technology,0000789019,2020.0,DEF 14A,2020-10-19,2020-12-02,0001193125-20-272025,d31295ddef14a.htm,...,266479,success,398,142,117,59,658,205,64,110


In [37]:
part2_df.to_csv(PART2_OUTPUT_PATH, index=False)

print(f"Saved Part 2 output to: {PART2_OUTPUT_PATH}")

Saved Part 2 output to: ../data/part2_proxy_disclosure_analysis.csv


In [39]:
part2_df = pd.read_csv(PART2_OUTPUT_PATH)

print(part2_df.shape)
part2_df.head()

(265, 24)


,ticker,company_name,sector,cik,year,document_type,filing_date,report_date,accession_number,primary_document,...,text_length,download_status,forward_looking_count,risk_language_count,diversity_inclusion,sustainability_environment,employees_human_capital,governance_ethics,community_social_impact,risk_security
0,AAPL,Apple,Technology,320193,2016.0,DEF 14A,2016-01-06,2016-02-26,0001193125-16-422528,d79474ddef14a.htm,...,293179,success,462,88,108,69,449,69,5,87
1,AAPL,Apple,Technology,320193,2017.0,DEF 14A,2017-01-06,2017-02-28,0001193125-17-003753,d257185ddef14a.htm,...,206517,success,189,92,130,12,421,70,17,96
2,AAPL,Apple,Technology,320193,2019.0,DEF 14A,2019-01-08,2019-03-01,0001193125-19-004664,d667873ddef14a.htm,...,175659,success,153,111,83,11,339,69,8,108
3,AAPL,Apple,Technology,320193,2020.0,DEF 14A,2020-01-03,2020-02-26,0001193125-20-001450,d799303ddef14a.htm,...,185553,success,187,120,74,51,372,79,7,123
4,AAPL,Apple,Technology,320193,2021.0,DEF 14A,2021-01-05,2021-02-23,0001193125-21-001987,d767770ddef14a.htm,...,189397,success,192,150,94,37,421,111,32,135


In [42]:
coverage_by_company = (
    part2_df
    .groupby(["ticker", "company_name", "sector"])["year"]
    .nunique()
    .reset_index(name="years_found")
    .sort_values("years_found")
)

In [43]:
print("Number of companies:", part2_df["ticker"].nunique())
print("Number of company-years:", len(part2_df))
print("Year range:", part2_df["year"].min(), "-", part2_df["year"].max())

Number of companies: 42
Number of company-years: 265
Year range: 2016.0 - 2024.0


In [44]:
theme_cols = [
    "diversity_inclusion",
    "sustainability_environment",
    "employees_human_capital",
    "governance_ethics",
    "community_social_impact",
    "risk_security"
]

extra_cols = [
    "forward_looking_count",
    "risk_language_count"
]

part2_norm = part2_df.copy()

for col in theme_cols + extra_cols:
    part2_norm[col + "_per_1000_words"] = (
        part2_norm[col] / part2_norm["word_count"].replace(0, pd.NA) * 1000
    )

part2_norm.head()

,ticker,company_name,sector,cik,year,document_type,filing_date,report_date,accession_number,primary_document,...,community_social_impact,risk_security,diversity_inclusion_per_1000_words,sustainability_environment_per_1000_words,employees_human_capital_per_1000_words,governance_ethics_per_1000_words,community_social_impact_per_1000_words,risk_security_per_1000_words,forward_looking_count_per_1000_words,risk_language_count_per_1000_words
0,AAPL,Apple,Technology,320193,2016.0,DEF 14A,2016-01-06,2016-02-26,0001193125-16-422528,d79474ddef14a.htm,...,5,87,2.330851,1.489155,9.690299,1.489155,0.107910,1.877630,9.970864,1.899212
1,AAPL,Apple,Technology,320193,2017.0,DEF 14A,2017-01-06,2017-02-28,0001193125-17-003753,d257185ddef14a.htm,...,17,96,4.092812,0.377798,13.254416,2.203822,0.535214,3.022385,5.950320,2.896452
2,AAPL,Apple,Technology,320193,2019.0,DEF 14A,2019-01-08,2019-03-01,0001193125-19-004664,d667873ddef14a.htm,...,8,108,3.061940,0.405799,12.505995,2.545468,0.295127,3.984211,5.644299,4.094883
3,AAPL,Apple,Technology,320193,2020.0,DEF 14A,2020-01-03,2020-02-26,0001193125-20-001450,d799303ddef14a.htm,...,7,123,2.586870,1.782843,13.004265,2.761658,0.244704,4.299797,6.537090,4.194924
4,AAPL,Apple,Technology,320193,2021.0,DEF 14A,2021-01-05,2021-02-23,0001193125-21-001987,d767770ddef14a.htm,...,32,135,3.251583,1.279878,14.562939,3.839635,1.106922,4.669826,6.641530,5.188696


In [45]:
PART2_NORM_OUTPUT_PATH = DATA_DIR / "part2_proxy_disclosure_analysis_normalized.csv"

part2_norm.to_csv(PART2_NORM_OUTPUT_PATH, index=False)

print(f"Saved normalized Part 2 data to: {PART2_NORM_OUTPUT_PATH}")

Saved normalized Part 2 data to: ../data/part2_proxy_disclosure_analysis_normalized.csv


In [47]:
norm_cols = [col + "_per_1000_words" for col in theme_cols + extra_cols]

sector_summary = (
    part2_norm
    .groupby("sector")[norm_cols + ["word_count"]]
    .mean()
    .reset_index()
)

sector_summary.to_csv(DATA_DIR / "part2_sector_summary.csv", index=False)

In [49]:
year_summary = (
    part2_norm
    .groupby("year")[norm_cols + ["word_count"]]
    .mean()
    .reset_index()
    .sort_values("year")
)

year_summary.to_csv(DATA_DIR / "part2_year_summary.csv", index=False)

In [50]:
company_summary = (
    part2_norm
    .groupby(["ticker", "company_name", "sector"])[norm_cols + ["word_count"]]
    .mean()
    .reset_index()
)

company_summary.head()

company_summary.sort_values(
    "diversity_inclusion_per_1000_words",
    ascending=False
).head(10)

company_summary.sort_values(
    "sustainability_environment_per_1000_words",
    ascending=False
).head(10)

company_summary.to_csv(DATA_DIR / "part2_company_summary.csv", index=False)

In [56]:
company_summary = (
    part2_norm
    .groupby(["ticker", "company_name", "sector"])[norm_cols + ["word_count"]]
    .mean()
    .reset_index()
)

company_summary.sort_values(
    "diversity_inclusion_per_1000_words",
    ascending=False
).head(10)

,ticker,company_name,sector,diversity_inclusion_per_1000_words,sustainability_environment_per_1000_words,employees_human_capital_per_1000_words,governance_ethics_per_1000_words,community_social_impact_per_1000_words,risk_security_per_1000_words,forward_looking_count_per_1000_words,risk_language_count_per_1000_words,word_count
3,AMZN,Amazon,Consumer Discretionary,6.706454,4.271486,14.673258,2.159152,2.872687,4.465906,6.486094,3.698305,70985.500000
17,INTC,Intel,Technology,5.225867,1.309107,14.217973,2.689781,0.958240,3.552928,11.201361,3.320938,76168.166667
28,ORCL,Oracle,Technology,5.128913,0.336476,14.667622,3.204443,0.098131,2.886156,8.693675,2.991976,53404.222222
33,SCHW,Charles Schwab,Financials,4.518205,0.608169,13.064992,2.781302,0.555670,3.581739,9.912463,3.548363,45330.250000
0,AAPL,Apple,Technology,4.407409,1.271211,14.030473,3.233074,0.989855,4.128577,7.675382,4.220556,38471.500000
15,HD,Home Depot,Consumer Discretionary,4.192950,0.728779,8.401028,2.804456,0.516252,3.682922,8.885198,3.144304,50716.142857
4,AVGO,Broadcom,Technology,3.984109,0.517173,12.740883,1.563404,0.016929,1.895111,10.025553,1.838384,43576.333333
5,AXP,American Express,Financials,3.941512,1.121669,9.831448,4.695690,1.146697,5.641364,10.063116,6.776376,53750.800000
9,CRM,Salesforce,Technology,3.747211,1.380551,12.696143,2.736450,0.862845,4.030717,12.277047,3.562316,81127.000000
25,MSFT,Microsoft,Technology,3.594961,2.314379,14.923851,4.829258,1.256329,4.803518,11.017278,4.809280,48173.000000
